# 04 - Évaluation (Option A)
BLEU-4, CIDEr, ROUGE-L, METEOR + VRAM, latence, taille. Figures et exemples dans `results/`.

In [ ]:
# Colab : activer le GPU T4. Remplacez <utilisateur> par votre nom GitHub.
!git clone https://github.com/<utilisateur>/vlm-quantization-project.git
%cd vlm-quantization-project
!pip install -q -r requirements.txt
!python data/download_flickr8k.py --n 500

In [ ]:
!apt-get -qq install -y default-jre-headless > /dev/null  # nécessaire pour METEOR

In [ ]:
import json, glob
import pandas as pd
from src.eval_captions import evaluate

meta = json.load(open('data/raw/metadata.json'))
rows, per_image = [], {}
for f in sorted(glob.glob('results/tables/resources_*.json')):
    r = json.load(open(f))
    tag = f"{r['model']}_{r['precision']}"
    metrics, per_img = evaluate(f'results/tables/captions_{tag}.csv', meta)
    rows.append({**r, **metrics})
    per_image[tag] = per_img
summary = pd.DataFrame(rows)
order = {'fp16': 0, 'int8': 1, 'int4': 2}
summary = summary.sort_values(['model', 'precision'], key=lambda s: s.map(order) if s.name == 'precision' else s)
summary.round(3).to_csv('results/tables/summary.csv', index=False)
summary.round(3)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, col, title in zip(axes, ['CIDEr', 'peak_vram_gb', 'latency_mean_s', 'model_size_gb'],
                          ['CIDEr (plus haut = mieux)', 'VRAM max (Go)', 'Latence moyenne (s)', 'Taille du modèle (Go)']):
    piv = summary.pivot(index='precision', columns='model', values=col).loc[['fp16', 'int8', 'int4']]
    piv.plot.bar(ax=ax, rot=0, title=title)
    ax.set_xlabel('')
plt.tight_layout()
plt.savefig('results/figures/quantization_overview.png', dpi=150)
plt.show()

In [ ]:
# Exemples visuels : 10 meilleures et 10 pires légendes (CIDEr par image) d'une configuration
from PIL import Image
tag = summary.sort_values('CIDEr', ascending=False).iloc[0][['model', 'precision']].astype(str).str.cat(sep='_')
print('Configuration illustrée :', tag)
pi = per_image[tag].sort_values('cider')
by_id = {m['id']: m for m in meta}
for name, sub in [('pires', pi.head(10)), ('meilleurs', pi.tail(10))]:
    fig, axes = plt.subplots(2, 5, figsize=(20, 9))
    for ax, (_, r) in zip(axes.ravel(), sub.iterrows()):
        ax.imshow(Image.open(by_id[int(r.id)]['file'])); ax.axis('off')
        ax.set_title(f"VLM : {r.caption[:55]}\nRéf : {by_id[int(r.id)]['references'][0][:55]}", fontsize=8)
    plt.suptitle(f'Exemples {name} ({tag})')
    plt.savefig(f'results/examples/exemples_{name}.png', dpi=120)
    plt.show()

In [ ]:
# Même image, 3 précisions : comparaison qualitative
model = summary['model'].iloc[0]
for i in [0, 1, 2]:
    print('Réf :', by_id[i]['references'][0])
    for p in ['fp16', 'int8', 'int4']:
        c = per_image[f'{model}_{p}'].set_index('id').loc[i, 'caption']
        print(f'  {p:5s}:', c)